# Capstone Project: Database Setup and Analysis
This notebook initializes the `bluestock_mf` SQLite database schema, verifies the created tables, and runs analytical queries.

In [1]:
# Global Configuration
DB_PATH = r"C:\Users\Sai Khedekar\Desktop\CapstoneProject_1\data\db\bluestock_mf.db"

## Step 1: Initialize Database Schema
This section reads the `schema.sql` file and executes it to create the necessary tables in the SQLite database.

In [2]:
import sqlite3

# Connect to the database and execute the schema script
conn = sqlite3.connect(DB_PATH)

with open("../sql/schema.sql", "r") as f:
    sql = f.read()

statements = sql.split(";")

for i, stmt in enumerate(statements):
    stmt = stmt.strip()
    if not stmt:
        continue

    try:
        conn.execute(stmt)
    except Exception as e:
        print(f"\nFAILED AT STATEMENT {i+1}")
        print(stmt[:500])
        print("\nERROR:", e)
        break

print("Schema created successfully!")

conn.commit()
conn.close()

Schema created successfully!


## Step 2: Verify Table Creation
Here, we check the `sqlite_master` table to confirm that all the required tables were generated successfully.

In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(DB_PATH)

# Query to list all tables
verification_query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tables_df = pd.read_sql(verification_query, conn)
conn.close()

# Display the tables found in the database
tables_df

,name
0,fund_master
1,nav_history
2,aum_history
3,sip_inflows
4,category_inflows
5,folio_count
6,scheme_performance
7,investor_transactions
8,portfolio_holdings
9,benchmark_indices


# Insert Data to Database

In [4]:
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path

# --------------------------------------------------
# PATHS
# --------------------------------------------------

DATA_DIR = Path("../data/processed")

engine = create_engine(
    f"sqlite:///{DB_PATH}"
)

# --------------------------------------------------
# CSV -> TABLE MAPPING
# --------------------------------------------------

files = {

    "clean_01_fund_master.csv":
        "fund_master",

    "clean_02_nav_history.csv":
        "nav_history",

    "clean_03_aum_by_fund_house.csv":
        "aum_history",

    "clean_04_monthly_sip_inflows.csv":
        "sip_inflows",

    "clean_05_category_inflows.csv":
        "category_inflows",

    "clean_06_industry_folio_count.csv":
        "folio_count",

    "clean_07_scheme_performance.csv":
        "scheme_performance",

    "clean_08_investor_transactions.csv":
        "investor_transactions",

    "clean_09_portfolio_holdings.csv":
        "portfolio_holdings",

    "clean_10_benchmark_indices.csv":
        "benchmark_indices"
}

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------

for file_name, table_name in files.items():

    path = DATA_DIR / file_name

    if not path.exists():
        print(f"Missing file: {file_name}")
        continue

    df = pd.read_csv(path)

    df.to_sql(
        table_name,
        engine,
        if_exists="append",
        index=False
    )

    print(
        f"Loaded {table_name:<25} {len(df):>8,} rows"
    )

print("\nDatabase load completed successfully.")

Loaded fund_master                     40 rows
Loaded nav_history                 46,000 rows
Loaded aum_history                     90 rows
Loaded sip_inflows                     48 rows
Loaded category_inflows               144 rows
Loaded folio_count                     21 rows
Loaded scheme_performance              40 rows
Loaded investor_transactions       32,778 rows
Loaded portfolio_holdings             322 rows
Loaded benchmark_indices            8,050 rows

Database load completed successfully.


## Step 3: Run Analytical Queries
Now that the data structure is ready, we run a query to find the **Top 5 Fund Houses** based on their Total Assets Under Management (AUM) in crores.

In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(DB_PATH)

# Query to fetch top 5 fund houses by AUM
analysis_query = """
ALTER TABLE nav_history 
ALTER COLUMN date DATE;
"""

top_funds_df = pd.read_sql(analysis_query, conn)
conn.close()

# Display the analysis result
top_funds_df